In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 1 — Bronze Layer
# MAGIC **Retail Pharmacy Chain — Shampoo Promotion Analysis**
# MAGIC
# MAGIC Ingest all 5 RAW CSV files into Delta Lake exactly as-is.
# MAGIC No cleaning. Audit metadata only.
# MAGIC
# MAGIC ```
# MAGIC RAW_Product.csv    →  bronze_product    (2 products + dirty dups)
# MAGIC RAW_Store.csv      →  bronze_store      (1,300 Canadian pharmacy stores)
# MAGIC RAW_Date.csv       →  bronze_date       (Year × Week calendar)
# MAGIC RAW_Promotion.csv  →  bronze_promotion  (price/flyer promotion types)
# MAGIC RAW_Sales.csv      →  bronze_sales      (store × product × week — fact)
# MAGIC ```
# MAGIC
# MAGIC > **Grain**: 1 row = 1 store × 1 product × 1 week.
# MAGIC > `SUM(Units, SalesAmt, GrossMargin, Transactions)` grouped by
# MAGIC > `Year, WeekNumber, Product` reproduces the original chain-level dataset.

# COMMAND ----------
# MAGIC %md ## 0. Configuration

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

current_user = spark.sql("SELECT current_user()").collect()[0][0]

# Keep RAW_PATH unchanged
# RAW_PATH = f"/Workspace/Users/{current_user}/raw_data"
RAW_PATH =f"/Workspace/Users/{current_user}/bda_course/Promotion_raw_data"
CATALOG = "workspace"
SCHEMA = "promotion"

# Create schema under catalog workspace
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog     : {CATALOG}")
print(f"Schema      : {SCHEMA}")
print(f"Raw path    : {RAW_PATH}")

# COMMAND ----------
# MAGIC %md ## 1. Helpers

# COMMAND ----------

def read_raw(filename):
    return (spark.read
            .option("header",      "true")
            .option("inferSchema", "false")   # Bronze: everything stays as STRING
            .option("multiLine",   "true")
            .option("escape",      '"')
            .csv(f"{RAW_PATH}/{filename}"))

def write_bronze(df, table_name):
    df_out = (df
        .withColumn("_ingest_time", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_batch_date",  F.lit(datetime.now().strftime("%Y-%m-%d")))
    )
    (df_out.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_{table_name}"))
    n = df_out.count()
    print(f"  bronze_{table_name:15s}  {n:>8,} rows")

# COMMAND ----------
# MAGIC %md ## 2. Dimension tables

# COMMAND ----------
# MAGIC %md ### 2.1 Product

# COMMAND ----------

df = read_raw("RAW_Product.csv")
print("Columns:", df.columns)
df.show(truncate=False)
write_bronze(df, "product")

# COMMAND ----------
# MAGIC %md ### 2.2 Store

# COMMAND ----------

df = read_raw("RAW_Store.csv")
print("Columns:", df.columns)
df.show(5, truncate=False)
write_bronze(df, "store")

# COMMAND ----------
# MAGIC %md ### 2.3 Date

# COMMAND ----------

df = read_raw("RAW_Date.csv")
print("Columns:", df.columns)
df.show(8, truncate=False)
write_bronze(df, "date")

# COMMAND ----------
# MAGIC %md ### 2.4 Promotion

# COMMAND ----------

df = read_raw("RAW_Promotion.csv")
print("Columns:", df.columns)
df.show(truncate=False)
write_bronze(df, "promotion")

# COMMAND ----------
# MAGIC %md ## 3. Fact table — Sales (store × product × week)

# COMMAND ----------

df = read_raw("RAW_Sales.csv")
print("Columns:", df.columns)
df.show(5, truncate=False)
write_bronze(df, "sales")

# COMMAND ----------
# MAGIC %md ## 4. Row count summary

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT 'product'   AS table_name, COUNT(*) AS rows FROM bronze_product
# MAGIC UNION ALL SELECT 'store',         COUNT(*)         FROM bronze_store
# MAGIC UNION ALL SELECT 'date',          COUNT(*)         FROM bronze_date
# MAGIC UNION ALL SELECT 'promotion',     COUNT(*)         FROM bronze_promotion
# MAGIC UNION ALL SELECT 'sales',         COUNT(*)         FROM bronze_sales
# MAGIC ORDER BY table_name

# COMMAND ----------
# MAGIC %md ## 5. Spot the data problems — do NOT fix here, that is Silver's job

# COMMAND ----------
# MAGIC %sql
# MAGIC -- Product: mixed Category/Supplier casing, duplicate rows
# MAGIC SELECT * FROM bronze_product

# COMMAND ----------
# MAGIC %sql
# MAGIC -- Promotion: Discount in 3 formats (0.10  /  "10%"  /  10.0),
# MAGIC --            mixed OnFlyer casing, mixed PromotionType casing, NULL DiscountTier
# MAGIC SELECT * FROM bronze_promotion

# COMMAND ----------
# MAGIC %sql
# MAGIC -- Date: FiscalYear in 3 formats (FY2021 / 2021 / FY-2021),
# MAGIC --       mixed Month casing, NULL Quarter
# MAGIC SELECT * FROM bronze_date LIMIT 15

# COMMAND ----------
# MAGIC %sql
# MAGIC -- Sales: mixed Product casing, mixed OnFlyer casing,
# MAGIC --        Discount in 3 formats, NULL Price, 50 duplicate rows
# MAGIC SELECT Year, WeekNumber, StoreID, Product, Price, OnFlyer, Discount,
# MAGIC        Units, SalesAmt, GrossMargin, Transactions
# MAGIC FROM bronze_sales
# MAGIC LIMIT 15

# COMMAND ----------
# MAGIC %sql
# MAGIC -- How many distinct Discount formats appear in sales?
# MAGIC SELECT Discount, COUNT(*) AS rows
# MAGIC FROM bronze_sales
# MAGIC GROUP BY Discount
# MAGIC ORDER BY rows DESC

# COMMAND ----------
# MAGIC %md
# MAGIC ## ✅ Bronze Complete
# MAGIC All 5 tables landed unchanged. Proceed to **Notebook 2 — Silver**.

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 2 — Silver Layer: Cleaning + Data Modeling
# MAGIC
# MAGIC **Steps:**
# MAGIC 1. Clean each Bronze dimension table
# MAGIC 2. Generate Surrogate Keys
# MAGIC 3. Join fact_sales to all dimensions — swap natural keys for surrogate keys
# MAGIC
# MAGIC **Star Schema output:**
# MAGIC ```
# MAGIC dim_product    (ProductKey)    ─┐
# MAGIC dim_store      (StoreKey)      ─┤
# MAGIC dim_date       (DateKey)       ─┼──► fact_sales
# MAGIC dim_promotion  (PromotionKey)  ─┘
# MAGIC ```
# MAGIC
# MAGIC **Key join challenge:**
# MAGIC `RAW_Sales` has no `PromotionName` column.
# MAGIC The promotion must be derived from `OnFlyer + Discount` — after cleaning both —
# MAGIC then joined to `dim_promotion` on `PromotionName`.

# COMMAND ----------
# MAGIC %md ## 0. Configuration

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "promotion"

def write_silver(df, table_name):
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_{table_name}"))
    n = df.count()
    print(f"  silver_{table_name:25s}  {n:>8,} rows")

# ── Shared discount cleaner (used in both dim_promotion and fact_sales) ──
# Handles 3 formats:  "10%"  →  0.10  |  10.0  →  0.10  |  0.10  →  0.10
def clean_discount(col_name):
    return (
        F.when(
            F.trim(F.col(col_name)).endswith("%"),
            F.regexp_extract(F.trim(col_name), r'([\d\.]+)', 1)
             .cast(DoubleType()) / 100
        ).when(
            F.col(col_name).cast(DoubleType()) > 1,
            F.col(col_name).cast(DoubleType()) / 100
        ).otherwise(
            F.col(col_name).cast(DoubleType())
        )
    )

# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 1. dim_product
# MAGIC **Natural key**: `Product` (name)
# MAGIC **Cleaning**: title-case all string cols, dedup
# MAGIC **Key**: `ProductKey` = row_number ordered by Product

# COMMAND ----------

df = spark.read.table(f"{CATALOG}.{SCHEMA}.bronze_product") \
         .drop("_ingest_time", "_source_file", "_batch_date")

df = (df
    .withColumn("Product",  F.initcap(F.trim("Product")))
    .withColumn("Brand",    F.initcap(F.trim("Brand")))
    .withColumn("Category", F.initcap(F.trim("Category")))
    .withColumn("Size",     F.trim("Size"))
    .withColumn("Supplier", F.initcap(F.trim("Supplier")))
    .withColumn("UnitCost", F.col("UnitCost").cast(DoubleType()))
    .dropDuplicates(["Product"])
    .withColumn("ProductKey", F.row_number().over(Window.orderBy("Product")))
    .select("ProductKey", "Product", "Brand", "Category", "Size", "Supplier", "UnitCost")
)

print("dim_product:")
df.show(truncate=False)
write_silver(df, "dim_product")

# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 2. dim_store
# MAGIC **Natural key**: `StoreID`
# MAGIC **Cleaning**: title-case City/Province, upper-case ProvinceAbbrev, dedup
# MAGIC **Key**: `StoreKey` = row_number ordered by StoreID

# COMMAND ----------

df = spark.read.table(f"{CATALOG}.{SCHEMA}.bronze_store") \
         .drop("_ingest_time", "_source_file", "_batch_date")

df = (df
    .withColumn("StoreID",         F.trim("StoreID"))
    .withColumn("StoreName",       F.initcap(F.trim("StoreName")))
    .withColumn("City",            F.initcap(F.trim("City")))
    .withColumn("Province",        F.initcap(F.trim("Province")))
    .withColumn("ProvinceAbbrev",  F.upper(F.trim("ProvinceAbbrev")))
    .withColumn("Country",         F.initcap(F.trim("Country")))
    .dropDuplicates(["StoreID"])
    .withColumn("StoreKey", F.row_number().over(Window.orderBy("StoreID")))
    .select("StoreKey", "StoreID", "StoreName", "City", "Province", "ProvinceAbbrev", "Country")
)

print(f"dim_store: {df.count():,} stores")
print("Province distribution:")
df.groupBy("Province").count().orderBy(F.desc("count")).show(15, truncate=False)
write_silver(df, "dim_store")

# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 3. dim_date
# MAGIC **Natural key**: `Year` + `WeekNumber`
# MAGIC **Cleaning**: standardise FiscalYear → "FY2021", title-case Month, fill NULL Quarter
# MAGIC **Key**: `DateKey` = Year × 100 + WeekNumber  →  202101

# COMMAND ----------

df = spark.read.table(f"{CATALOG}.{SCHEMA}.bronze_date") \
         .drop("_ingest_time", "_source_file", "_batch_date")

df = (df
    .withColumn("Year",        F.col("Year").cast(IntegerType()))
    .withColumn("WeekNumber",  F.col("WeekNumber").cast(IntegerType()))
    .withColumn("MonthNumber", F.col("MonthNumber").cast(IntegerType()))
    # Standardise FiscalYear: "FY2021" / "2021" / "FY-2021"  →  "FY2021"
    .withColumn("FiscalYear",
        F.concat(F.lit("FY"),
                 F.regexp_extract(F.col("FiscalYear"), r'(\d{4})', 1)))
    # Standardise Month casing
    .withColumn("Month", F.initcap(F.trim("Month")))
    # Fill NULL Quarter from MonthNumber
    # Retail quarters: Q1=Feb-Apr, Q2=May-Jul, Q3=Aug-Oct, Q4=Nov-Jan
    .withColumn("Quarter",
        F.when(F.col("Quarter").isNotNull(), F.col("Quarter"))
         .when(F.col("MonthNumber").isin([2,3,4]),   F.lit("Q1"))
         .when(F.col("MonthNumber").isin([5,6,7]),   F.lit("Q2"))
         .when(F.col("MonthNumber").isin([8,9,10]),  F.lit("Q3"))
         .otherwise(F.lit("Q4")))
    .dropDuplicates(["Year", "WeekNumber"])
    # DateKey = YYYYWW integer
    .withColumn("DateKey",
        (F.col("Year") * 100 + F.col("WeekNumber")).cast(IntegerType()))
    .select("DateKey", "Year", "WeekNumber", "WeekStartDate",
            "Month", "MonthNumber", "Quarter", "FiscalYear")
)

print("dim_date sample:")
df.orderBy("DateKey").show(10, truncate=False)
write_silver(df, "dim_date")

# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 4. dim_promotion
# MAGIC **Natural key**: `PromotionName`
# MAGIC **Cleaning**: fix Discount format, standardise casing, fill NULL DiscountTier
# MAGIC **Key**: `PromotionKey` = row_number ordered by PromotionName
# MAGIC
# MAGIC > The cleaned `PromotionName` + `OnFlyer` + `Discount` will be used
# MAGIC > in the fact join to identify which promotion applied to each sales row.

# COMMAND ----------

df = spark.read.table(f"{CATALOG}.{SCHEMA}.bronze_promotion") \
         .drop("_ingest_time", "_source_file", "_batch_date")

df = (df
    .withColumn("Discount",      clean_discount("Discount"))
    .withColumn("PromotionName", F.trim("PromotionName"))
    .withColumn("OnFlyer",       F.initcap(F.trim("OnFlyer")))
    .withColumn("PromotionType", F.initcap(F.trim("PromotionType")))
    .withColumn("DiscountTier",  F.initcap(F.trim("DiscountTier")))
    # Fill NULL DiscountTier from Discount value
    .withColumn("DiscountTier",
        F.when(F.col("DiscountTier").isNotNull(), F.col("DiscountTier"))
         .when(F.col("Discount") >= 0.30, F.lit("Deep"))
         .when(F.col("Discount") >  0.00, F.lit("Mid"))
         .otherwise(F.lit("None")))
    .dropDuplicates(["PromotionName"])
    .withColumn("PromotionKey", F.row_number().over(Window.orderBy("PromotionName")))
    .select("PromotionKey", "PromotionName", "OnFlyer", "Discount",
            "PromotionType", "DiscountTier")
)

print("dim_promotion:")
df.orderBy("Discount", "OnFlyer").show(20, truncate=False)
write_silver(df, "dim_promotion")

# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 5. fact_sales
# MAGIC
# MAGIC This is the core modeling step.
# MAGIC `RAW_Sales` has no surrogate keys and no `PromotionName`.
# MAGIC We must:
# MAGIC 1. Clean `Product`, `OnFlyer`, `Discount`, cast numeric columns, fill NULL `Price`
# MAGIC 2. **Derive `PromotionName`** from cleaned `OnFlyer + Discount` (same logic as dim_promotion)
# MAGIC 3. Join all 4 dimensions to get surrogate keys
# MAGIC 4. Recalculate `SalesAmt` and `GrossMargin` for rows where `Price` was NULL
# MAGIC
# MAGIC **Join chain:**
# MAGIC ```
# MAGIC RAW_Sales
# MAGIC   + dim_product   on  Product                     → ProductKey
# MAGIC   + dim_store     on  StoreID                     → StoreKey
# MAGIC   + dim_date      on  Year + WeekNumber           → DateKey
# MAGIC   + dim_promotion on  derived PromotionName       → PromotionKey
# MAGIC ```

# COMMAND ----------

df = spark.read.table(f"{CATALOG}.{SCHEMA}.bronze_sales") \
         .drop("_ingest_time", "_source_file", "_batch_date")

# Load only the key-lookup columns needed for joining
dim_prod  = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_product") \
                 .select("ProductKey", "Product", "UnitCost")
dim_store = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_store") \
                 .select("StoreKey", "StoreID")
dim_date  = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_date") \
                 .select("DateKey", "Year", "WeekNumber")
dim_promo = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_promotion") \
                 .select("PromotionKey", "PromotionName")

# ── Step 1: Clean sales ───────────────────────────────────────────

df = (df
    # Standardise Product casing to match dim_product
    .withColumn("Product",    F.initcap(F.trim("Product")))
    # Standardise OnFlyer casing
    .withColumn("OnFlyer",    F.initcap(F.trim("OnFlyer")))
    # Fix Discount to decimal (same 3-format logic as dim_promotion)
    .withColumn("Discount",   clean_discount("Discount"))
    # Cast dimension join keys
    .withColumn("Year",       F.col("Year").cast(IntegerType()))
    .withColumn("WeekNumber", F.col("WeekNumber").cast(IntegerType()))
    .withColumn("StoreID",    F.trim("StoreID"))
    # Cast measures
    .withColumn("Price",      F.col("Price").cast(DoubleType()))
    .withColumn("Units",      F.col("Units").cast(IntegerType()))
    .withColumn("SalesAmt",      F.col("SalesAmt").cast(DoubleType()))
    .withColumn("GrossMargin",   F.col("GrossMargin").cast(DoubleType()))
    .withColumn("Transactions",  F.col("Transactions").cast(IntegerType()))
    # Remove duplicates on natural key
    .dropDuplicates(["Year", "WeekNumber", "StoreID", "Product"])
)

# ── Step 2: Derive PromotionName (same formula used to build dim_promotion) ──
#
# Examples after cleaning:
#   OnFlyer=No,  Discount=0.00  →  "No Promotion"
#   OnFlyer=Yes, Discount=0.30  →  "30% Off + Flyer"
#   OnFlyer=No,  Discount=0.35  →  "35% Off"

df = df.withColumn(
    "PromotionName",
    F.when(
        F.col("Discount") == 0,
        F.lit("No Promotion")
    ).when(
        F.col("OnFlyer") == "Yes",
        F.concat(
            (F.col("Discount") * 100).cast(IntegerType()).cast(StringType()),
            F.lit("% Off + Flyer")
        )
    ).otherwise(
        F.concat(
            (F.col("Discount") * 100).cast(IntegerType()).cast(StringType()),
            F.lit("% Off")
        )
    )
)

# ── Step 3: Join all 4 dimension keys ─────────────────────────────

df_fact = (df
    .join(dim_prod,  on="Product",               how="left")
    .join(dim_store, on="StoreID",               how="left")
    .join(dim_date,  on=["Year", "WeekNumber"],  how="left")
    .join(dim_promo, on="PromotionName",         how="left")
)

# ── Step 4: Fix NULL Price rows ───────────────────────────────────
# Where Price was NULL in RAW, recalculate from SalesAmt / Units
# (SalesAmt = Price × Units was verified in source)
df_fact = df_fact.withColumn(
    "Price",
    F.when(
        F.col("Price").isNull(),
        F.round(F.col("SalesAmt") / F.col("Units"), 2)
    ).otherwise(F.col("Price"))
)

# Flag below-cost promotions (negative GrossMargin — loss leaders)
df_fact = df_fact.withColumn(
    "IsBelowCost",
    F.when(F.col("GrossMargin") < 0, F.lit(1)).otherwise(F.lit(0))
)

# ── Step 5: Select final columns ──────────────────────────────────
df_fact = df_fact.select(
    "ProductKey",
    "StoreKey",
    "DateKey",
    "PromotionKey",
    "Year",
    "WeekNumber",
    F.col("Price")        .alias("ActualPrice"),
    F.col("Discount")     .alias("DiscountPct"),
    F.col("Units")        .alias("UnitsSold"),
    F.col("SalesAmt")     .alias("SalesAmt"),
    F.col("GrossMargin")  .alias("GrossMargin"),
    F.col("Transactions") .alias("Transactions"),
    "IsBelowCost",
)

# ── Step 6: Join quality check ─────────────────────────────────────
print("=== Join Quality Check ===")
total = df_fact.count()
print(f"Total rows         : {total:,}")
print(f"NULL ProductKey    : {df_fact.filter(F.col('ProductKey').isNull()).count():,}")
print(f"NULL StoreKey      : {df_fact.filter(F.col('StoreKey').isNull()).count():,}")
print(f"NULL DateKey       : {df_fact.filter(F.col('DateKey').isNull()).count():,}")
print(f"NULL PromotionKey  : {df_fact.filter(F.col('PromotionKey').isNull()).count():,}")
print(f"Below-cost rows    : {df_fact.filter(F.col('IsBelowCost')==1).count():,}")

write_silver(df_fact, "fact_sales")

# COMMAND ----------
# MAGIC %md ## 6. Verify: aggregate back to chain level

# COMMAND ----------
# MAGIC %sql
# MAGIC -- Re-aggregate store rows → should match original chain-level numbers
# MAGIC SELECT
# MAGIC     d.Year,
# MAGIC     d.WeekNumber,
# MAGIC     p.Product,
# MAGIC     pr.PromotionName,
# MAGIC     ROUND(AVG(f.ActualPrice),   2)  AS Price,
# MAGIC     SUM(f.UnitsSold)               AS TotalUnits,
# MAGIC     ROUND(SUM(f.SalesAmt),     2)  AS TotalSalesAmt,
# MAGIC     ROUND(SUM(f.GrossMargin),  2)  AS TotalGrossMargin,
# MAGIC     SUM(f.Transactions)            AS TotalTransactions
# MAGIC FROM silver_fact_sales     f
# MAGIC JOIN silver_dim_product    p  ON f.ProductKey   = p.ProductKey
# MAGIC JOIN silver_dim_date       d  ON f.DateKey      = d.DateKey
# MAGIC JOIN silver_dim_promotion  pr ON f.PromotionKey = pr.PromotionKey
# MAGIC GROUP BY d.Year, d.WeekNumber, p.Product, pr.PromotionName
# MAGIC ORDER BY d.Year, d.WeekNumber, p.Product
# MAGIC LIMIT 20

# COMMAND ----------
# MAGIC %md ## 7. Star Schema row counts

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT 'dim_product'   AS table_name, COUNT(*) AS rows FROM silver_dim_product
# MAGIC UNION ALL SELECT 'dim_store',         COUNT(*) FROM silver_dim_store
# MAGIC UNION ALL SELECT 'dim_date',          COUNT(*) FROM silver_dim_date
# MAGIC UNION ALL SELECT 'dim_promotion',     COUNT(*) FROM silver_dim_promotion
# MAGIC UNION ALL SELECT 'fact_sales',        COUNT(*) FROM silver_fact_sales

# COMMAND ----------
# MAGIC %md
# MAGIC ## ✅ Silver Complete
# MAGIC ```
# MAGIC dim_product    ProductKey   ─┐
# MAGIC dim_store      StoreKey     ─┤
# MAGIC dim_date       DateKey      ─┼──► fact_sales  (152,521 rows)
# MAGIC dim_promotion  PromotionKey ─┘
# MAGIC ```
# MAGIC Proceed to **Notebook 3 — Gold**.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 3 — Gold Layer: Promotion Analysis
# MAGIC
# MAGIC Each Gold table directly answers one or more project questions:
# MAGIC
# MAGIC | Gold Table | Project Question |
# MAGIC |------------|-----------------|
# MAGIC | `gold_price_elasticity`   | Q1/Q2: Which price maximises units/margin? |
# MAGIC | `gold_promotion_uplift`   | Q5/Q6/Q7: Discount impact vs baseline |
# MAGIC | `gold_weekly_trend`       | Q3: Is shampoo seasonal? |
# MAGIC | `gold_province_summary`   | Chain-level + province breakdown |
# MAGIC | `gold_loss_leader`        | Q9/Q10: Is Aussie @$2.49 an effective loss leader? |

# COMMAND ----------
# MAGIC %md ## 0. Configuration

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "promotion"

# Load Silver tables once — reused across all Gold builds
fact   = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_fact_sales")
d_prod = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_product")
d_store= spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_store")
d_date = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_date")
d_promo= spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_promotion")

def write_gold(df, table_name):
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_{table_name}"))
    n = df.count()
    print(f"  gold_{table_name:30s}  {n:>6,} rows")

# MAGIC %md
# MAGIC ---
# MAGIC ## 1. gold_price_elasticity
# MAGIC **Answers Q1 & Q2**: Which price point maximises units sold? Which maximises gross margin?
# MAGIC
# MAGIC Aggregates all store-weeks at each price point to get chain totals,
# MAGIC then calculates average weekly performance per price.

# COMMAND ----------

df = (fact
    .join(d_prod .select("ProductKey","Product","UnitCost"),                   "ProductKey")
    .join(d_promo.select("PromotionKey","PromotionName","OnFlyer","DiscountTier"), "PromotionKey")
    # Chain-level aggregate first: one row per product × week × price point
    .groupBy("ProductKey","Product","UnitCost",
             "PromotionKey","PromotionName","OnFlyer","DiscountTier",
             "ActualPrice","DiscountPct",F.col("Year"),"WeekNumber")
    .agg(
        F.sum("UnitsSold")        .alias("ChainUnits"),
        F.sum("SalesAmt")         .alias("ChainSalesAmt"),
        F.sum("GrossMargin")      .alias("ChainGrossMargin"),
        F.sum("Transactions")     .alias("ChainTransactions"),
        F.sum("IsBelowCost")      .alias("StoresBelowCost"),
        F.count("StoreKey")       .alias("StoresActive"),
    )
    # Now average across all weeks at this price point
    .groupBy("ProductKey","Product","UnitCost",
             "PromotionName","OnFlyer","DiscountTier","ActualPrice","DiscountPct")
    .agg(
        F.count ("WeekNumber")             .alias("WeeksAtThisPrice"),
        F.round(F.avg("ChainUnits"),    0) .alias("AvgWeeklyUnits"),
        F.round(F.sum("ChainUnits"),    0) .alias("TotalUnits"),
        F.round(F.avg("ChainSalesAmt"), 2) .alias("AvgWeeklySales"),
        F.round(F.sum("ChainSalesAmt"), 2) .alias("TotalSales"),
        F.round(F.avg("ChainGrossMargin"),2).alias("AvgWeeklyMargin"),
        F.round(F.sum("ChainGrossMargin"),2).alias("TotalMargin"),
        F.round(F.avg("ChainTransactions"),0).alias("AvgWeeklyTransactions"),
        F.round(F.avg("StoresActive"),  0) .alias("AvgStoresActive"),
    )
    .withColumn("GrossMarginPct",
        F.round(F.col("TotalMargin") / F.col("TotalSales") * 100, 1))
    .withColumn("MarginPerUnit",
        F.round(F.col("AvgWeeklyMargin") / F.col("AvgWeeklyUnits"), 4))
    # Rank within product: by units, by margin
    .withColumn("RankByUnits",
        F.rank().over(Window.partitionBy("Product").orderBy(F.desc("AvgWeeklyUnits"))))
    .withColumn("RankByMargin",
        F.rank().over(Window.partitionBy("Product").orderBy(F.desc("AvgWeeklyMargin"))))
    .withColumn("_gold_ts", F.current_timestamp())
    .orderBy("Product", "ActualPrice")
)

print("Price elasticity — top rows:")
df.select("Product","ActualPrice","OnFlyer","DiscountPct","WeeksAtThisPrice",
          "AvgWeeklyUnits","AvgWeeklyMargin","GrossMarginPct",
          "RankByUnits","RankByMargin").show(25, truncate=False)
write_gold(df, "price_elasticity")

# MAGIC %md
# MAGIC ---
# MAGIC ## 2. gold_promotion_uplift
# MAGIC **Answers Q5, Q6, Q7**: Uplift of each promotion vs regular-price baseline.
# MAGIC Includes the specific 25% and 60% scenarios (Q5/Q6).
# MAGIC
# MAGIC > Baseline = average chain weekly units/margin when Discount = 0.

# COMMAND ----------

# Calculate baseline per product (no-promotion weeks only)
baseline = (fact
    .join(d_promo.select("PromotionKey","Discount"), "PromotionKey")
    .filter(F.col("Discount") == 0)
    .join(d_date.select("DateKey",F.col("Year").alias("DateYear"),F.col("WeekNumber").alias("DateWeekNumber")), "DateKey")
    # Chain aggregate first
    .groupBy("ProductKey","DateYear","DateWeekNumber")
    .agg(F.sum("UnitsSold").alias("WeekUnits"),
         F.sum("SalesAmt").alias("WeekSales"),
         F.sum("GrossMargin").alias("WeekMargin"))
    # Then average across weeks
    .groupBy("ProductKey")
    .agg(
        F.round(F.avg("WeekUnits"),  0).alias("BaselineUnits"),
        F.round(F.avg("WeekSales"),  2).alias("BaselineSales"),
        F.round(F.avg("WeekMargin"), 2).alias("BaselineMargin"),
    )
)

df = (fact
    .join(d_prod .select("ProductKey","Product"),                                            "ProductKey")
    .join(d_promo.select("PromotionKey","PromotionName","OnFlyer","Discount","DiscountTier"),"PromotionKey")
    .join(d_date.select("DateKey",F.col("Year").alias("DateYear"),F.col("WeekNumber").alias("DateWeekNumber")), "DateKey")
    # Chain aggregate: one row per product × promo × week
    .groupBy("ProductKey","Product","PromotionKey","PromotionName",
             "OnFlyer","Discount","DiscountTier","DateYear","DateWeekNumber")
    .agg(F.sum("UnitsSold")   .alias("WeekUnits"),
         F.sum("SalesAmt")    .alias("WeekSales"),
         F.sum("GrossMargin") .alias("WeekMargin"),
         F.sum("IsBelowCost") .alias("StoresBelowCost"))
    # Average across all weeks this promo ran
    .groupBy("ProductKey","Product","PromotionKey","PromotionName",
             "OnFlyer","Discount","DiscountTier")
    .agg(
        F.count ("DateWeekNumber")              .alias("WeeksRan"),
        F.round(F.avg("WeekUnits"),    0)   .alias("AvgWeeklyUnits"),
        F.round(F.avg("WeekSales"),    2)   .alias("AvgWeeklySales"),
        F.round(F.avg("WeekMargin"),   2)   .alias("AvgWeeklyMargin"),
        F.round(F.sum("WeekMargin"),   2)   .alias("TotalMargin"),
        F.sum("StoresBelowCost")            .alias("BelowCostInstances"),
    )
    .join(baseline, "ProductKey", "left")
    # Uplift calculations
    .withColumn("UnitUpliftPct",
        F.round((F.col("AvgWeeklyUnits")  - F.col("BaselineUnits"))
                / F.col("BaselineUnits") * 100, 1))
    .withColumn("SalesUpliftPct",
        F.round((F.col("AvgWeeklySales")  - F.col("BaselineSales"))
                / F.col("BaselineSales") * 100, 1))
    .withColumn("MarginUpliftPct",
        F.round((F.col("AvgWeeklyMargin") - F.col("BaselineMargin"))
                / F.col("BaselineMargin") * 100, 1))
    .withColumn("IncrementalUnits",
        F.round(F.col("AvgWeeklyUnits") - F.col("BaselineUnits"), 0))
    .withColumn("IncrementalMargin",
        F.round(F.col("AvgWeeklyMargin") - F.col("BaselineMargin"), 2))
    .withColumn("_gold_ts", F.current_timestamp())
    .orderBy("Product", "Discount")
)

print("Promotion uplift:")
df.select("Product","PromotionName","OnFlyer","Discount",
          "AvgWeeklyUnits","BaselineUnits","UnitUpliftPct",
          "AvgWeeklyMargin","BaselineMargin","MarginUpliftPct",
          "BelowCostInstances").show(25, truncate=False)
write_gold(df, "promotion_uplift")

# MAGIC %md
# MAGIC ---
# MAGIC ## 3. gold_weekly_trend
# MAGIC **Answers Q3**: Is shampoo seasonal?
# MAGIC Weekly chain-level units/sales/margin with 4-week moving average to smooth noise.

# COMMAND ----------

df = (fact
    .join(d_prod .select("ProductKey","Product"),               "ProductKey")
    .join(d_date .select("DateKey",F.col("Year").alias("DateYear"),F.col("WeekNumber").alias("DateWeekNumber"),"Month","MonthNumber","Quarter","FiscalYear","WeekStartDate"), "DateKey")
    .join(d_promo.select("PromotionKey","PromotionName","OnFlyer","Discount"),                 "PromotionKey")
    # Chain aggregate per product × week
    .groupBy("Product","DateYear","DateWeekNumber","Month","MonthNumber",
             "Quarter","FiscalYear","WeekStartDate","PromotionName","OnFlyer","Discount")
    .agg(
        F.sum ("UnitsSold")         .alias("ChainUnits"),
        F.round(F.sum("SalesAmt"),2).alias("ChainSalesAmt"),
        F.round(F.sum("GrossMargin"),2).alias("ChainGrossMargin"),
        F.sum ("Transactions")      .alias("ChainTransactions"),
        F.sum ("IsBelowCost")       .alias("StoresBelowCost"),
        F.count("StoreKey")         .alias("StoreCount"),
    )
    .withColumn("DateKey",
        (F.col("DateYear") * 100 + F.col("DateWeekNumber")).cast("int"))
    # 4-week rolling average of units (per product)
    .withColumn("RollingAvg4WkUnits",
        F.round(
            F.avg("ChainUnits").over(
                Window.partitionBy("Product")
                      .orderBy("DateKey")
                      .rowsBetween(-3, 0)   # current + 3 prior weeks
            ), 0
        )
    )
    # Week-over-week change
    .withColumn("PrevWeekUnits",
        F.lag("ChainUnits", 1).over(
            Window.partitionBy("Product").orderBy("DateKey")))
    .withColumn("WoWChangePct",
        F.round((F.col("ChainUnits") - F.col("PrevWeekUnits"))
                / F.col("PrevWeekUnits") * 100, 1))
    .drop("PrevWeekUnits")
    .withColumn("GrossMarginPct",
        F.round(F.col("ChainGrossMargin") / F.col("ChainSalesAmt") * 100, 1))
    .withColumn("IsPromoWeek",
        F.when(F.col("Discount") > 0, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("_gold_ts", F.current_timestamp())
    .orderBy("Product","DateYear","DateWeekNumber")
)

write_gold(df, "weekly_trend")

# MAGIC %md
# MAGIC ---
# MAGIC ## 4. gold_province_summary
# MAGIC Chain-level + province breakdown for regional analysis.
# MAGIC Supports: which provinces respond most to promotions?

# COMMAND ----------

df = (fact
    .join(d_prod .select("ProductKey","Product"),                                             "ProductKey")
    .join(d_store.select("StoreKey","Province","ProvinceAbbrev"),                            "StoreKey")
    .join(d_date .select("DateKey",F.col("Year").alias("DateYear"),F.col("WeekNumber").alias("DateWeekNumber"),"FiscalYear","Quarter"), "DateKey")
    .join(d_promo.select("PromotionKey","PromotionName","OnFlyer","Discount","DiscountTier"),"PromotionKey")
    .groupBy("Product","Province","ProvinceAbbrev","DateYear","DateWeekNumber","FiscalYear",
             "Quarter","PromotionName","OnFlyer","Discount","DiscountTier")
    .agg(
        F.sum ("UnitsSold")              .alias("TotalUnits"),
        F.round(F.sum("SalesAmt"),    2) .alias("TotalSales"),
        F.round(F.sum("GrossMargin"), 2) .alias("TotalMargin"),
        F.sum ("Transactions")           .alias("TotalTransactions"),
        F.count("StoreKey")              .alias("StoreCount"),
        F.sum ("IsBelowCost")            .alias("BelowCostInstances"),
    )
    .withColumn("GrossMarginPct",
        F.round(F.col("TotalMargin") / F.col("TotalSales") * 100, 1))
    .withColumn("UnitsPerStore",
        F.round(F.col("TotalUnits") / F.col("StoreCount"), 1))
    .withColumn("_gold_ts", F.current_timestamp())
    .orderBy("Product","Province","DateYear","DateWeekNumber","Quarter")
)

write_gold(df, "province_summary")

# MAGIC %md
# MAGIC ---
# MAGIC ## 5. gold_loss_leader
# MAGIC **Answers Q9 & Q10**: Is Aussie @ $2.49 an effective loss leader?
# MAGIC Compares the $2.49 promo week directly to all other Aussie price points.

# COMMAND ----------

df = (fact
    .join(d_prod .select("ProductKey","Product","UnitCost"),                                 "ProductKey")
    .join(d_promo.select("PromotionKey","PromotionName","OnFlyer","Discount","DiscountTier"),"PromotionKey")
    .join(d_date.select("DateKey",F.col("Year").alias("DateYear"),F.col("WeekNumber").alias("DateWeekNumber")), "DateKey")
    # Chain aggregate per week
    .groupBy("ProductKey","Product","UnitCost","PromotionKey","PromotionName",
             "OnFlyer","Discount","DiscountTier","DateYear","DateWeekNumber","ActualPrice")
    .agg(
        F.sum ("UnitsSold")              .alias("ChainUnits"),
        F.round(F.sum("SalesAmt"),    2) .alias("ChainSales"),
        F.round(F.sum("GrossMargin"), 2) .alias("ChainMargin"),
        F.sum ("Transactions")           .alias("ChainTransactions"),
        F.count("StoreKey")              .alias("StoreCount"),
    )
    # Average by price/promo tier
    .groupBy("Product","UnitCost","PromotionName","OnFlyer",
             "Discount","DiscountTier","ActualPrice")
    .agg(
        F.count ("DateWeekNumber")               .alias("WeeksObserved"),
        F.round(F.avg("ChainUnits"),      0) .alias("AvgWeeklyUnits"),
        F.round(F.avg("ChainSales"),      2) .alias("AvgWeeklySales"),
        F.round(F.avg("ChainMargin"),     2) .alias("AvgWeeklyMargin"),
        F.round(F.avg("ChainTransactions"),0).alias("AvgWeeklyTransactions"),
        F.round(F.avg("StoreCount"),      0) .alias("AvgStoresActive"),
    )
    # Cost and margin detail
    .withColumn("TotalCostPerWeek",
        F.round(F.col("UnitCost") * F.col("AvgWeeklyUnits"), 2))
    .withColumn("GrossMarginPct",
        F.round(F.col("AvgWeeklyMargin") / F.col("AvgWeeklySales") * 100, 1))
    .withColumn("MarginPerUnit",
        F.round(F.col("AvgWeeklyMargin") / F.col("AvgWeeklyUnits"), 4))
    .withColumn("IsLossLeader",
        F.when(F.col("AvgWeeklyMargin") < 0, F.lit(1)).otherwise(F.lit(0)))
    # Filter to Aussie only for this table
    .filter(F.col("Product") == "Aussie")
    .withColumn("_gold_ts", F.current_timestamp())
    .orderBy("ActualPrice")
)

print("Aussie loss leader analysis:")
df.select("ActualPrice","OnFlyer","Discount","WeeksObserved",
          "AvgWeeklyUnits","AvgWeeklySales","AvgWeeklyMargin",
          "GrossMarginPct","IsLossLeader").show(20, truncate=False)
write_gold(df, "loss_leader")

# MAGIC %md ## 6. Summary

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT 'price_elasticity' AS gold_table, COUNT(*) AS rows FROM gold_price_elasticity
# MAGIC UNION ALL SELECT 'promotion_uplift', COUNT(*) FROM gold_promotion_uplift
# MAGIC UNION ALL SELECT 'weekly_trend',     COUNT(*) FROM gold_weekly_trend
# MAGIC UNION ALL SELECT 'province_summary', COUNT(*) FROM gold_province_summary
# MAGIC UNION ALL SELECT 'loss_leader',      COUNT(*) FROM gold_loss_leader

# MAGIC %sql
# MAGIC -- Q1: Best price for units (per product)
# MAGIC SELECT Product, ActualPrice, OnFlyer, DiscountPct,
# MAGIC        AvgWeeklyUnits, RankByUnits, AvgWeeklyMargin, RankByMargin
# MAGIC FROM gold_price_elasticity
# MAGIC WHERE RankByUnits <= 5
# MAGIC ORDER BY Product, RankByUnits

# MAGIC %sql
# MAGIC -- Q2: Best price for margin (per product)
# MAGIC SELECT Product, ActualPrice, OnFlyer, DiscountPct,
# MAGIC        AvgWeeklyMargin, GrossMarginPct, RankByMargin, RankByUnits
# MAGIC FROM gold_price_elasticity
# MAGIC WHERE RankByMargin <= 5
# MAGIC ORDER BY Product, RankByMargin

# MAGIC %sql
# MAGIC -- Q4: Cost per unit of each product
# MAGIC SELECT Product, UnitCost
# MAGIC FROM silver_dim_product
# MAGIC ORDER BY Product

# MAGIC %sql
# MAGIC -- Q7: On-Flyer vs No-Flyer impact — same discount, compare with/without flyer
# MAGIC SELECT Product, Discount, OnFlyer, PromotionName,
# MAGIC        AvgWeeklyUnits, UnitUpliftPct, AvgWeeklyMargin, MarginUpliftPct
# MAGIC FROM gold_promotion_uplift
# MAGIC WHERE Discount > 0
# MAGIC ORDER BY Product, Discount, OnFlyer

# MAGIC %sql
# MAGIC -- Q9/Q10: Aussie loss leader detail
# MAGIC SELECT ActualPrice, OnFlyer, WeeksObserved,
# MAGIC        AvgWeeklyUnits, AvgWeeklySales,
# MAGIC        AvgWeeklyMargin, GrossMarginPct, IsLossLeader
# MAGIC FROM gold_loss_leader
# MAGIC ORDER BY ActualPrice

# MAGIC %md
# MAGIC ## ✅ Gold Complete
# MAGIC
# MAGIC | Table | Answers |
# MAGIC |-------|---------|
# MAGIC | `gold_price_elasticity` | Q1: best price for units, Q2: best price for margin |
# MAGIC | `gold_promotion_uplift` | Q5: 25% discount impact, Q6: 60% discount, Q7: flyer effect |
# MAGIC | `gold_weekly_trend`     | Q3: seasonality analysis (rolling avg, WoW change) |
# MAGIC | `gold_province_summary` | Regional performance, province-level breakdown |
# MAGIC | `gold_loss_leader`      | Q9: Is Aussie $2.49 effective? Q10: 2-for-$5 scenario |